# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook guides you through loading and exploring the FAIR^2 dataset using the `mlcroissant` library. You will learn how to:

- Load Croissant metadata and records from the dataset
- Inspect the available record sets and fields (referenced by their `@id`)
- Extract tabular data into Pandas DataFrames
- Conduct basic exploratory data analysis (EDA) and visualization

## Dataset Source
This dataset describes mass spectrometry analysis of protein abundance, modifications, and sequences derived from human samples. It is made available as a Croissant schema here:

- Schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load the metadata and inspect basic information about the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Print basic metadata info
print("Title:", dataset.metadata.name)
print("Description:", dataset.metadata.description)


## 2. Data Overview
Explore the available **record sets** and their **fields/columns** using their `@id`. This is the foundation to extract data correctly in later steps.


In [ ]:
# List all record sets by their @id and name
print('RecordSets available:')
for rset in dataset.record_sets:
    print(f"  @id: {rset.id}  name: {rset.name}")
    print("    Fields and Columns:")
    for field in rset.fields:
        colid_descr = f"-> column @id: {field.column.id}" if hasattr(field, 'column') and hasattr(field.column,'id') else ''
        print(f"      field @id: {field.id}  name: {field.name} {colid_descr}")


> 🔎 **Tip:** The above cell lists the machine-readable `@id`s you should use to refer to specific record sets and fields in all subsequent analyses.


## 3. Data Extraction
Load data from all record sets into Pandas DataFrames. You can refer to a DataFrame by the record set `@id` key. Adjust the code below to use specific `@id`s from the overview.


In [ ]:
# Build a list of record set @id strings
record_set_ids = [rs.id for rs in dataset.record_sets]

# Create a dictionary of DataFrames, one per record set
dataframes = {}

for rs in dataset.record_sets:
    print(f"Loading records from RecordSet @id: {rs.id} ...")
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"  Loaded {len(df)} rows, columns: {list(df.columns)}\n")
# For demonstration, pick the first record set for further EDA and visualization
selected_record_set_id = record_set_ids[0] if record_set_ids else None
if selected_record_set_id:
    df = dataframes[selected_record_set_id]
    print(f"First rows from record set @id: {selected_record_set_id}")
    display(df.head())


## 4. Exploratory Data Analysis (EDA)
Let's demonstrate basic EDA such as filtering, normalization, and grouping on a numeric field. Adjust `numeric_field_id` and `group_field_id` to `@id`s of fields as relevant.

> Example below: Pick an integer-like numeric column (e.g., 'mw' for molecular weight, or another quantitative variable) from the listing above.

In [ ]:
from IPython.display import display

# Change this to a real field/column `@id` as needed (see Data Overview above)
numeric_field_id = None
group_field_id = None

df = dataframes[selected_record_set_id]
print(f"Columns in record set {selected_record_set_id}:", list(df.columns))

# Try to auto-select a numeric field as an example
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break

if numeric_field_id is None:
    print("No numeric field was found for automatic analysis.")
else:
    print(f"Selected numeric field: {numeric_field_id}")
    # Example filter: keep only rows with value > threshold
    threshold = df[numeric_field_id].quantile(0.5)
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} above median ({threshold:.2f})")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()

    print(f"Normalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to choose a group field (categorical/text)
    group_field_candidates = [c for c in df.columns if pd.api.types.is_string_dtype(df[c]) and c != numeric_field_id]
    group_field_id = group_field_candidates[0] if group_field_candidates else None
    if group_field_id:
        print(f"Grouping by field: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        display(grouped_df.head())
    else:
        print('No suitable grouping field found for demonstration.')


## 5. Visualization
Visualize the distribution of the selected numeric field and (optionally) compare it across groups if available.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of field: {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id:
        plt.figure(figsize=(12, 5))
        top_groups = df[group_field_id].value_counts().index[:5]
        sns.boxplot(data=df[df[group_field_id].isin(top_groups)], x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id} (top 5 categories)")
        plt.show()


## 6. Conclusion
- In this notebook you learned how to load, inspect, and explore a Croissant dataset using the `mlcroissant` Python library.
- By referencing all entities (record sets, fields, columns) by their `@id`, your analyses are robust and schema-aware.

You can now adapt and extend this workflow for your own FAIR-compliant data science projects using other datasets described in Croissant!